### Manual: 02-system-probe

Run this cell in the Kaggle notebook UI. Environment persists across cells in the same session (scripts in `/kaggle/working/smoke_test`, venv, repo clone under `/root/bevfusion_smoke`).


In [ ]:
import os, sys
WORK = os.environ.get('SMOKE_WORK', '/kaggle/working/smoke_test')
if not os.path.exists(os.path.join(WORK, 'smoke_common.py')):
    raise SystemExit('Run 01-bootstrap first in this session (or run the full notebook once to create scripts).')
sys.path.insert(0, WORK)
print('guard ok:', WORK)


In [ ]:
"""System probe: GPU, driver, CUDA toolkits, compilers, dataset mount.

Records what the build cell has to work with. No downloads, no installs.
"""

import glob
import json
import os
import re
import shutil
import sys

WORK = os.environ.get("SMOKE_WORK", "/kaggle/working/smoke_test")
sys.path.insert(0, WORK)
import smoke_common as sc  # noqa: E402

report = {"python": sys.version, "executable": sys.executable}
report["os_release"] = sc.capture("grep -E '^(NAME|VERSION)=' /etc/os-release")
report["kernel"] = sc.capture("uname -r")
report["cpu_count"] = os.cpu_count()
report["mem_total"] = sc.capture("free -g | sed -n 2p")
report["disk"] = sc.capture("df -h /kaggle/working /root | sed 's/  */ /g'")

report["nvidia_smi"] = sc.capture(
    "nvidia-smi --query-gpu=index,name,driver_version,memory.total,compute_cap "
    "--format=csv,noheader"
)

report["gcc_default"] = sc.capture("gcc --version 2>/dev/null | head -1")
report["gxx_default"] = sc.capture("g++ --version 2>/dev/null | head -1")
report["compilers"] = sc.capture("ls /usr/bin/gcc* /usr/bin/g++* 2>/dev/null | tr '\\n' ' '")
report["uv"] = shutil.which("uv") or sc.capture("command -v uv")
report["sys_python"] = sys.version.split()[0]

# --------------------------------------------------------------- CUDA toolkits
candidates = []
for line in sc.capture("ls -d /usr/local/cuda* 2>/dev/null").splitlines():
    line = line.strip()
    if line:
        candidates.append(line)
for extra in glob.glob("/usr/local/cuda*/bin/nvcc") + glob.glob("/usr/bin/nvcc") + glob.glob(
    "/usr/lib/cuda/bin/nvcc"
):
    candidates.append(os.path.dirname(os.path.dirname(extra)))
which_nvcc = shutil.which("nvcc")
if which_nvcc:
    candidates.append(os.path.dirname(os.path.dirname(which_nvcc)))

toolkits = []
for base in sorted(set(candidates)):
    nvcc = os.path.join(base, "bin", "nvcc")
    if not os.path.isfile(nvcc):
        continue
    raw = sc.capture('"%s" --version' % nvcc)
    match = re.search(r"release (\d+\.\d+)", raw)
    toolkits.append(
        {
            "path": base,
            "nvcc": nvcc,
            "release": match.group(1) if match else None,
            "raw": raw.splitlines()[-1] if raw else None,
            "cudart_header": os.path.isfile(os.path.join(base, "include", "cuda_runtime.h")),
        }
    )
report["cuda_toolkits_on_disk"] = toolkits
report["CUDA_HOME_env"] = os.environ.get("CUDA_HOME")
report["cuda_version_json"] = sc.capture("cat /usr/local/cuda*/version.json 2>/dev/null | head -20")
report["cuda_apt_sources"] = sc.capture(
    "grep -rh '^deb ' /etc/apt/sources.list /etc/apt/sources.list.d/ 2>/dev/null | grep -i cuda"
)
report["cuda_keyring"] = os.path.isfile("/usr/share/keyrings/cuda-archive-keyring.gpg")

# ------------------------------------------------------------------- dataset
root = sc.dataset_root()
report["dataset"] = {"root": root}
if root:
    report["dataset"]["version_dirs"] = sc.capture('ls "%s" | tr "\\n" " "' % root)
    infos = []
    for pattern in ("nuscenes_infos_train.pkl", "nuscenes_infos_val.pkl", "nuscenes_infos_test.pkl"):
        infos.append({"file": pattern, "exists": os.path.isfile(os.path.join(root, pattern))})
    report["dataset"]["infos"] = infos
    report["dataset"]["real_sample_possible"] = any(item["exists"] for item in infos)

print(json.dumps(report, indent=2, sort_keys=True))
sc.save_json("system.json", report)
sc.update(system=report, cuda_toolkits=toolkits)
print("PROBE_OK")
